In the previous notebook we trained a T5 model to summarize xsum articles. What we did during training is backpropagation, which updates all the parameters at every step of our model. This can be quite memory-heavy and inefficient. So what better approach is there?

The answer is LoRA! The concept of LoRA is quite simple. Imagine we have a weight matrix $W$ with dimensions (512, 512) -> 262,144 weights. Backprop needs to store the gradient for every single weight and update them, quite tough.

What LoRA does instead is it freezes our W matrix, meaning no more backprop for those weights, and instead picks two smaller matrices $A$ (512, 4) and $B$ (4, 512). Together they have 4,096 parameters, but multiplying them results in a (512, 512) matrix. In this case 4 is what we call the ***rank***.

Now during training instead of updating the gradients of $W$, we update the gradients of $A$ and $B$, and simply do $output = W * input + A * B * input$.

But why does this work? We literally reduced the number of params by 98,5%, how can this still function properly? The answer is surprisingly simple: T5 is a pre-trained model, that already has general knowledge of what words mean, and how they relate to each other. During training we simply want the model to go from general to suitable for our specific task, which doesn't require as many parameters.

So LoRA is memory, storage and speed efficient, then where are the drawbacks? There certainly are cons, such as some tasks not being suitable for such fine-tuning (translating from english to chinese for example) or the full fine-tuning performing sightly better than the LoRA model sometimes, but for our goals LoRA seems perfect.

In [ ]:
# To use LoRA we need to install peft (Parameter Efficient Fine-Tuning)
!pip install peft

In [1]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

In [2]:
# Select the tokenizer and the model
tokenizer = T5Tokenizer.from_pretrained('t5-small')
model = T5ForConditionalGeneration.from_pretrained('t5-small')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [32]:
from datasets import load_dataset
train_ds = load_dataset('xsum', split='train[:5000]') # import the first 1k training set items
val_ds = load_dataset('xsum', split='validation[:1000]') # import the first 200 validation set items

In [33]:
def preprocess(example): # Example is a dict containing the document, its summary label and an id
  '''
  function to preprocess text sequences to feed into the model
  '''
  text = 'summarize: ' + example['document'] # Add the summarize command
  inputs_document = tokenizer(text, max_length=512, truncation=True) # Tokenize the text
  inputs_label = tokenizer(example['summary'], max_length=512, truncation=True) # Tokenize the target

  preprocessed = {
      'input_ids' : inputs_document['input_ids'],
      'attention_mask' : inputs_document['attention_mask'],
      'labels' : inputs_label['input_ids']
  }

  return preprocessed

In [34]:
# apply the preprocess function to the datasets (add 3 extra features in the dicts input_ids, attention_mask, labels)
train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
!pip install --upgrade torchao

In [ ]:
!pip install --upgrade torch

In [ ]:
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8, # Rank
    target_modules=["q","v"], # target the Q and V matrices in each attention head
    task_type="SEQ_2_SEQ_LM", # seq2seq task
)

peft_model = get_peft_model(model, config)
peft_model.to('cuda')

In [36]:
peft_model.print_trainable_parameters()

trainable params: 294,912 || all params: 60,801,536 || trainable%: 0.4850


As we can see here, peft froze 60.8M params, and we instead only work with ~300k

In [37]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model) # Data Collator is used to add padding to the batch, to match the longest item's length

In [38]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

# set up the model trainer's hyperparams
training_args = Seq2SeqTrainingArguments( # Setup the hyperparams of the model
    output_dir='./results',
    num_train_epochs=3, # train for 3 epochs
    per_device_train_batch_size=8, # handle 8 sequences per batch
    learning_rate=3e-4, # higher lr with LoRA
    eval_strategy='epoch', # evaluate the model's performance after every epoch
    predict_with_generate=True, # set to true, so model also generates text
)

# Set up the model trainer
trainer = Seq2SeqTrainer(
    model=peft_model, # model -> peft_model
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator
)

In [39]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.125833,2.712446


Epoch,Training Loss,Validation Loss
1,3.125833,2.712446
2,2.966918,2.680951
3,2.938555,2.675746


TrainOutput(global_step=1875, training_loss=2.9937083984375, metrics={'train_runtime': 530.3418, 'train_samples_per_second': 28.284, 'train_steps_per_second': 3.535, 'total_flos': 2039499272749056.0, 'train_loss': 2.9937083984375, 'epoch': 3.0})

In [27]:
import torch

In [40]:
# Check out an example
example = val_ds[0]

inputs = tokenizer.decode(example['input_ids'], skip_special_tokens=True) # encode the tokens into ids
print("Input:", inputs[:200])

# Use the trained model here to generate a sequence
outputs = model.generate(torch.tensor([example['input_ids']]).to(model.device), max_length=60, num_beams=4, no_repeat_ngram_size=3) # pass ids to decoder to generate text
print("LoRA Fine-tuned output:", tokenizer.decode(outputs[0], skip_special_tokens=True)) # transform token ids from the output to tokens

Input: summarize: The ex-Reading defender denied fraudulent trading charges relating to the Sodje Sports Foundation - a charity to raise money for Nigerian sport. Mr Sodje, 37, is jointly charged with elder 
LoRA Fine-tuned output: Two brothers, Sam and Stephen Sodje, have been charged with fraudulent trading.


In [18]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=a3bdc55bc4593bf413a0d909b9f0406f6655e9a0547d27862740dabd12a446a4
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [29]:
from rouge_score import rouge_scorer

In [30]:
# define what scores we will be using
scorer = rouge_scorer.RougeScorer(rouge_types=['rouge1', 'rouge2', 'rougeL'])

# Function to compute the 3 different rouge scores
def compute_rouge(prediction, reference):
  scores = scorer.score(prediction, reference)
  return scores

In [41]:
avg_f1_score_rouge1 = 0
avg_f1_score_rouge2 = 0
avg_f1_score_rougeL = 0

for i in range(len(val_ds)):
  val_example = val_ds[i]['document'] # input text
  val_reference = val_ds[i]['summary'] # label
  val_inputs = tokenizer('summarize: ' + val_example, return_tensors='pt', max_length=512, truncation=True).to(model.device) # encode the input
  val_output = model.generate(val_inputs['input_ids'], max_length=50) # decode the tokens given by the encoder
  val_summary = tokenizer.decode(val_output[0], skip_special_tokens=True) # token_ids -> tokens

  results = compute_rouge(val_summary, val_reference)

  # Compute the average f1 scores
  avg_f1_score_rouge1 += results['rouge1'].fmeasure / len(val_ds)
  avg_f1_score_rouge2 += results['rouge2'].fmeasure / len(val_ds)
  avg_f1_score_rougeL += results['rougeL'].fmeasure / len(val_ds)

print(f'Average Rouge1 F1: {avg_f1_score_rouge1:.5f}')
print(f'Average Rouge2 F1: {avg_f1_score_rouge2:.5f}')
print(f'Average RougeL F1: {avg_f1_score_rougeL:.5f}')

Average Rouge1 F1: 0.25638
Average Rouge2 F1: 0.06003
Average RougeL F1: 0.19676


Here I did 2 things, increased the training data from 1000 -> 5000, and used LoRA to decrease the number of parameters, but got better results than the previous 'general' model.

Previous model scores:

Average Rouge1 F1: 0.20258

Average Rouge2 F1: 0.04190

Average RougeL F1: 0.16057

Current model scores:

Average Rouge1 F1: 0.25638 (26.5% better)

Average Rouge2 F1: 0.06003 (43.3% better)

Average RougeL F1: 0.19676 (22.5% better)

Not the best results, but still a pretty good improvement.